# 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
from pathlib import Path

ROOT = '/content/drive/MyDrive/Projects/ml-group-project'
DATA = os.path.join(ROOT, 'data')
ML_32M_DATA = os.path.join(DATA, 'ml-32m')

PARQUETS = Path(os.path.join(DATA, 'parquets'))
PARQUETS.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = PARQUETS / "train.parquet"
VAL_PATH = PARQUETS / "val.parquet"
PREDICT_PATH = PARQUETS / "test.parquet"

ENCODER_PATH = PARQUETS / "encoder"

In [ ]:
import numpy as np

np.random.seed(42)

In [ ]:
!pip install replay-rec pyspark==3.5.1 lightning torchvision --upgrade --force-reinstall

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 8.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 11.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 7.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 368.1/368.1 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 158.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 1.5 MB/s eta 0:00

# 2. Dataset

- [MovieLens 32M dataset](https://grouplens.org/datasets/movielens/)
- Load dataframes

In [ ]:
import pandas as pd

ratings = pd.read_csv(os.path.join(ML_32M_DATA, 'ratings.csv'))
ratings.head(3)

,userId,movieId,rating,timestamp
0,1,17,4.0,944249077
1,1,25,1.0,944250228
2,1,29,2.0,943230976


In [ ]:
ratings["timestamp"] = ratings["timestamp"].astype("int64")
ratings = ratings.sort_values(by="timestamp")
ratings["timestamp"] = ratings.groupby("userId").cumcount()
ratings.head(3)

,userId,movieId,rating,timestamp
3994589,25062,1176,4.0,0
4958006,30917,1079,3.0,0
4957963,30917,47,5.0,1


In [ ]:
from replay.preprocessing import LabelEncoder, LabelEncodingRule

encoder = LabelEncoder(
    [
        LabelEncodingRule("userId"),
        LabelEncodingRule("movieId"),
    ]
)
encoded_interactions = encoder.fit_transform(ratings)
encoded_interactions.head()

,rating,timestamp,userId,movieId
0,4.0,0,25061,1148
1,3.0,0,30916,1052
2,5.0,1,30916,46
3,3.0,2,30916,20
4,5.0,0,38834,10


In [ ]:
from replay.splitters import LastNSplitter

# Rename columns to match replay-rec's default expectations
# (user_id, item_id, timestamp)
encoded_interactions = encoded_interactions.rename(
    columns={'userId': 'user_id', 'movieId': 'item_id'}
)

splitter = LastNSplitter(
    N=1,
    divide_column="user_id", # Use the renamed column
    query_column="user_id",  # Use the renamed column
    strategy="interactions",
    drop_cold_users=True,
    drop_cold_items=True
)

test_events, test_gt = splitter.split(encoded_interactions)
validation_events, validation_gt = splitter.split(test_events)
train_events = validation_events

In [ ]:
from replay.data.nn.utils import groupby_sequences

def bake_data(full_data):
    return groupby_sequences(events=full_data, groupby_col="user_id", sort_col="timestamp")

In [ ]:
train_events = bake_data(train_events)

validation_events = bake_data(validation_events)
validation_gt = bake_data(validation_gt)

test_events = bake_data(test_events)
test_gt = bake_data(test_gt)

train_events.head()

,user_id,rating,timestamp,item_id
0,0,"[4.0, 1.0, 4.0, 2.0, 1.0, 5.0, 5.0, 1.0, 5.0, ...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[2905, 2874, 2798, 2985, 2790, 536, 820, 1108,..."
1,1,"[4.0, 5.0, 1.0, 3.0, 1.0, 5.0, 3.0, 5.0, 2.0, ...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[584, 375, 292, 344, 339, 580, 151, 587, 228, ..."
2,2,"[3.0, 1.0, 4.0, 3.5, 4.0, 3.5, 4.0, 4.0, 3.5, ...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[1923, 461, 2177, 166, 1489, 4202, 1440, 2526,..."
3,3,"[2.0, 3.0, 3.0, 3.0, 5.0, 4.0, 3.0, 2.0, 2.0, ...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[1746, 1179, 2653, 1293, 2025, 1239, 2591, 260..."
4,4,"[4.0, 3.0, 3.0, 5.0, 1.0, 4.0, 3.0, 4.0, 3.0, ...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[584, 582, 148, 375, 292, 163, 580, 344, 339, ..."


In [ ]:
def add_gt_to_events(events_df, gt_df):
    gt_to_join = gt_df[["user_id", "item_id"]].rename(columns={"item_id": "ground_truth"})

    events_df = events_df.merge(gt_to_join, on="user_id", how="inner")
    return events_df

In [ ]:
validation_events = add_gt_to_events(validation_events, validation_gt)
test_events = add_gt_to_events(test_events, test_gt)

In [ ]:
train_events.to_parquet(TRAIN_PATH)
validation_events.to_parquet(VAL_PATH)
test_events.to_parquet(PREDICT_PATH)

encoder.save(ENCODER_PATH)

/usr/local/lib/python3.12/dist-packages/replay/preprocessing/label_encoder.py:975: UserWarning: There is already LabelEncoder object saved at the given path. File will be overwrited.
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/replay/preprocessing/label_encoder.py:537: UserWarning: There is already LabelEncodingRule object saved at the given path. File will be overwrited.
  warnings.warn(msg)


# 3. Model

In [ ]:
MAX_SEQ_LEN = 50
NUM_BLOCKS = 2
NUM_HEADS = 2
DROPOUT = 0.3
BATCH_SIZE = 32
EMBEDDING_DIM = 64

In [ ]:
from replay.data import FeatureHint, FeatureType
from replay.data.nn.schema import TensorFeatureInfo, TensorSchema

import torch
import lightning as L
from replay.nn.transform.template import make_default_sasrec_transforms

from replay.data.nn import ParquetModule

from replay.nn.sequential import SasRec
from replay.nn.lightning import LightningModule

NUM_UNIQUE_ITEMS = len(encoder.mapping["movieId"])

tensor_schema = TensorSchema(
    [
        TensorFeatureInfo(
            name="item_id",
            is_seq=True,
            padding_value=NUM_UNIQUE_ITEMS,
            cardinality=NUM_UNIQUE_ITEMS,
            embedding_dim=EMBEDDING_DIM,
            feature_type=FeatureType.CATEGORICAL,
            feature_hint=FeatureHint.ITEM_ID,
        )
    ]
)

transforms = make_default_sasrec_transforms(tensor_schema)

train_metadata = {
    "train": {
        "item_id": {"shape": MAX_SEQ_LEN + 1, "padding": tensor_schema["item_id"].padding_value},
    },
    "validate": {
        "item_id": {"shape": MAX_SEQ_LEN, "padding": tensor_schema["item_id"].padding_value},
        "ground_truth": {"shape": 1, "padding": -1}
    },
}

parquet_module = ParquetModule(
    train_path=TRAIN_PATH,
    validate_path=VAL_PATH,
    batch_size=BATCH_SIZE,
    metadata=train_metadata,
    transforms=transforms,
)

sasrec = SasRec.from_params(
    schema=tensor_schema,
    embedding_dim=EMBEDDING_DIM,
    max_sequence_length=MAX_SEQ_LEN,
    num_heads=NUM_HEADS,
    num_blocks=NUM_BLOCKS,
    dropout=DROPOUT,
)
model = LightningModule(sasrec)

/tmp/ipykernel_20483/3083315375.py:41: UserWarning: The following dataset paths aren't provided: test,predict. Make sure to disable these stages in your Lightning Trainer configuration.
  parquet_module = ParquetModule(


In [ ]:
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping
from lightning.pytorch.loggers import CSVLogger
from replay.nn.lightning.callback import ComputeMetricsCallback
import lightning as L
import torch

torch.set_float32_matmul_precision("medium")

csv_logger = CSVLogger(save_dir=f"{ROOT}/sasrec/logs/train", name="SasRec-example")

checkpoint_callback = ModelCheckpoint(
    dirpath=f"{ROOT}/sasrec/checkpoints",
    save_top_k=1,
    monitor="train_loss_epoch",
    mode="min",
    verbose=True,
)

early_stop_callback = EarlyStopping(
    monitor="train_loss_epoch",
    mode="min",
    patience=2,
    min_delta=0.001,
)

trainer = L.Trainer(
    max_epochs=50,
    callbacks=[checkpoint_callback, early_stop_callback],
    logger=csv_logger,
    enable_progress_bar=True,
    enable_model_summary=True,
    log_every_n_steps=50,
)

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


# 4. Train

In [ ]:
trainer.fit(model, datamodule=parquet_module)

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /content/drive/MyDrive/Projects/ml-group-project/sasrec/checkpoints exists and is not empty.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type   ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ SasRec │  5.5 M │ train │     0 │
└───┴───────┴────────┴────────┴───────┴───────┘

Trainable params: 5.5 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 5.5 M                                                                                                
Total estimated model params size (MB): 21                                                                         
Modules in train mode: 40                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/data.py:123: Your `IterableDataset` has 
`__len__` defined. In combination with multi-process data loading (when num_workers > 1), `__len__` could be 
inaccurate if each worker is not configured independently to avoid having duplicate data.

INFO: Epoch 0, global step 6280: 'train_loss_epoch' reached 6.79419 (best 6.79419), saving model to '/content/drive/MyDrive/Projects/ml-group-project/sasrec/checkpoints/epoch=0-step=6280.ckpt' as top 1
INFO:lightning.pytorch.utilities.rank_zero:Epoch 0, global step 6280: 'train_loss_epoch' reached 6.79419 (best 6.79419), saving model to '/content/drive/MyDrive/Projects/ml-group-project/sasrec/checkpoints/epoch=0-step=6280.ckpt' as top 1
INFO: Epoch 1, global step 12560: 'train_loss_epoch' reached 6.47412 (best 6.47412), saving model to '/content/drive/MyDrive/Projects/ml-group-project/sasrec/checkpoints/epoch=1-step=12560.ckpt' as top 1
INFO:lightning.pytorch.utilities.rank_zero:Epoch 1, global step 12560: 'train_loss_epoch' reached 6.47412 (best 6.47412), saving model to '/content/drive/MyDrive/Projects/ml-group-project/sasrec/checkpoints/epoch=1-step=12560.ckpt' as top 1
INFO: Epoch 2, global step 18840: 'train_loss_epoch' reached 6.43590 (best 6.43590), saving model to '/content/dri

# 5. Evaluate

In [ ]:
best_model_path = checkpoint_callback.best_model_path
best_epoch = int(best_model_path.split("epoch=")[1].split("-")[0])

best_model_path, best_epoch

('/content/drive/MyDrive/Projects/ml-group-project/sasrec/checkpoints/epoch=8-step=56520.ckpt',
 8)

In [ ]:
sasrec = SasRec.from_params(
    schema=tensor_schema,
    embedding_dim=EMBEDDING_DIM,
    max_sequence_length=MAX_SEQ_LEN,
    num_heads=NUM_HEADS,
    num_blocks=NUM_BLOCKS,
    dropout=DROPOUT,
    excluded_features=None,
)

best_model = LightningModule.load_from_checkpoint(best_model_path, model=sasrec)
best_model.eval();

In [ ]:
inference_metadata = {
    "predict": {
        "user_id": {},
        "item_id": {"shape": MAX_SEQ_LEN, "padding": tensor_schema["item_id"].padding_value},
    }
}

parquet_module = ParquetModule(
    predict_path=PREDICT_PATH,
    batch_size=BATCH_SIZE,
    metadata=inference_metadata,
    transforms=transforms,
)

/tmp/ipykernel_20483/3991935412.py:8: UserWarning: The following dataset paths aren't provided: train,validate,test. Make sure to disable these stages in your Lightning Trainer configuration.
  parquet_module = ParquetModule(


In [ ]:
from replay.nn.lightning.callback import PandasTopItemsCallback

csv_logger = CSVLogger(save_dir="sasrec/logs/test", name="SasRec-example")

TOPK = [1, 5, 10, 20]

pandas_prediction_callback = PandasTopItemsCallback(
    top_k=max(TOPK),
    query_column="user_id",
    item_column="item_id",
    rating_column="score",
)

trainer = L.Trainer(callbacks=[pandas_prediction_callback], logger=csv_logger, inference_mode=True)
trainer.predict(best_model, datamodule=parquet_module, return_predictions=False)

pandas_res = pandas_prediction_callback.get_result()
pandas_res

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/data.py:123: Your `IterableDataset` has `__len__` defined. In combination with multi-process data loading (when num_workers > 1), `__len__` could be inaccurate if each worker is not configured independently to avoid having duplicate data.


,user_id,item_id,score
0,0,972,1.920056
0,0,1154,1.802484
0,0,1018,1.554739
0,0,1322,1.508173
0,0,1144,1.47876
...,...,...,...
200850,200947,12429,5.222558
200850,200947,12324,5.147133
200850,200947,12331,5.144228
200850,200947,12536,5.089415


In [ ]:
from replay.metrics import MAP, OfflineMetrics, Precision, Recall
from replay.metrics.torch_metrics_builder import metrics_to_df

result_metrics = OfflineMetrics(
    [Recall(TOPK), Precision(TOPK), MAP(TOPK)],
    query_column="user_id",
    rating_column="score",
)(pandas_res, test_gt.explode("item_id"))

metrics_to_df(result_metrics)

/usr/local/lib/python3.12/dist-packages/replay/metrics/offline_metrics.py:375: UserWarning: ground_truth contains queries that are not presented in recommendations
  warnings.warn(f"{dataset_name} contains queries that are not presented in recommendations")


k,1,5,10,20
MAP,0.027239,0.051077,0.058273,0.063414
Precision,0.027239,0.019420,0.015199,0.011359
Recall,0.027239,0.097102,0.151993,0.227183


In [ ]:
encoder = LabelEncoder.load(ENCODER_PATH)
pandas_res_renamed = pandas_res.rename(columns={'user_id': 'userId', 'item_id': 'movieId'})
inversed_res = encoder.inverse_transform(pandas_res_renamed)
inversed_res.head()

,userId,movieId,score
0,1,994,1.920056
0,1,1183,1.802484
0,1,1041,1.554739
0,1,1357,1.508173
0,1,1172,1.47876


In [ ]:
import os
from pathlib import Path
import pandas as pd
import torch
import lightning as L
from lightning.pytorch.loggers import CSVLogger

from replay.preprocessing import LabelEncoder
from replay.data import FeatureHint, FeatureType
from replay.data.nn.schema import TensorFeatureInfo, TensorSchema
from replay.nn.transform.template import make_default_sasrec_transforms
from replay.data.nn import ParquetModule
from replay.nn.sequential import SasRec
from replay.nn.lightning import LightningModule
from replay.nn.lightning.callback import PandasTopItemsCallback

# --- Paths & Configuration ---
ROOT = '/content/drive/MyDrive/Projects/ml-group-project'
DATA = os.path.join(ROOT, 'data')
PARQUETS = Path(os.path.join(DATA, 'parquets'))
ENCODER_PATH = PARQUETS / "encoder"
PREDICT_PATH = PARQUETS / "test.parquet"

# Checkpoint path obtained from the training phase
CHECKPOINT_PATH = f"{ROOT}/sasrec/checkpoints/epoch=8-step=56520.ckpt"

# Model parameters
MAX_SEQ_LEN = 50
NUM_BLOCKS = 2
NUM_HEADS = 2
DROPOUT = 0.3
BATCH_SIZE = 32
EMBEDDING_DIM = 64
TOP_K = 20

torch.set_float32_matmul_precision("medium")

# --- 1. Load Encoder & Recreate Schema ---
encoder = LabelEncoder.load(ENCODER_PATH)
NUM_UNIQUE_ITEMS = len(encoder.mapping["movieId"])

tensor_schema = TensorSchema(
    [
        TensorFeatureInfo(
            name="item_id",
            is_seq=True,
            padding_value=NUM_UNIQUE_ITEMS,
            cardinality=NUM_UNIQUE_ITEMS,
            embedding_dim=EMBEDDING_DIM,
            feature_type=FeatureType.CATEGORICAL,
            feature_hint=FeatureHint.ITEM_ID,
        )
    ]
)
transforms = make_default_sasrec_transforms(tensor_schema)

# --- 2. Setup DataModule ---
inference_metadata = {
    "predict": {
        "user_id": {},
        "item_id": {"shape": MAX_SEQ_LEN, "padding": tensor_schema["item_id"].padding_value},
    }
}

parquet_module = ParquetModule(
    predict_path=PREDICT_PATH,
    batch_size=BATCH_SIZE,
    metadata=inference_metadata,
    transforms=transforms,
)

# --- 3. Load Model ---
sasrec = SasRec.from_params(
    schema=tensor_schema,
    embedding_dim=EMBEDDING_DIM,
    max_sequence_length=MAX_SEQ_LEN,
    num_heads=NUM_HEADS,
    num_blocks=NUM_BLOCKS,
    dropout=DROPOUT,
)
model = LightningModule.load_from_checkpoint(CHECKPOINT_PATH, model=sasrec)
model.eval()

# --- 4. Run Prediction ---
pandas_prediction_callback = PandasTopItemsCallback(
    top_k=TOP_K,
    query_column="user_id",
    item_column="item_id",
    rating_column="score",
)

trainer = L.Trainer(
    callbacks=[pandas_prediction_callback],
    logger=CSVLogger(save_dir=f"{ROOT}/sasrec/logs/inference", name="SasRec-standalone-inference"),
    inference_mode=True,
    enable_progress_bar=True
)

trainer.predict(model, datamodule=parquet_module, return_predictions=False)

# --- 5. Retrieve & Inverse Transform Results ---
pandas_res = pandas_prediction_callback.get_result()
pandas_res_renamed = pandas_res.rename(columns={'user_id': 'userId', 'item_id': 'movieId'})
final_recommendations = encoder.inverse_transform(pandas_res_renamed)

# Display the top 20 candidates for the first user
print("\nTop 20 candidates generated successfully:")
final_recommendations.head(20)


/tmp/ipykernel_20483/828845317.py:65: UserWarning: The following dataset paths aren't provided: train,validate,test. Make sure to disable these stages in your Lightning Trainer configuration.
  parquet_module = ParquetModule(
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/data.py:123: Your `IterableDataset` has `__len__` defined. In combination with multi-process data loading (when num_workers > 1), `__len__` could be inaccurate if each worker is not configured independently to avoid having duplicate data.



Top 20 candidates generated successfully:


,userId,movieId,score
0,1,994,1.920056
0,1,1183,1.802484
0,1,1041,1.554739
0,1,1357,1.508173
0,1,1172,1.47876
0,1,1094,1.407261
0,1,1185,1.341836
0,1,1354,1.311287
0,1,1132,1.258398
0,1,1296,1.240132
